# 7. Swarm

*Using Microsoft Semantic Kernel (Agent Framework)*

Demonstrates a swarm architecture where agents can dynamically hand off tasks to each other. This pattern enables flexible agent collaboration with agents autonomously deciding when to transfer control to more suitable agents.

In [ ]:
import os
from typing import Annotated
from dotenv import load_dotenv
import semantic_kernel as sk
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import kernel_function
from semantic_kernel.connectors.ai.open_ai.prompt_execution_settings.azure_chat_prompt_execution_settings import (
    AzureChatPromptExecutionSettings,
)
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior

load_dotenv()

kernel = sk.Kernel()
service_id = "chat-gpt"
kernel.add_service(
    AzureChatCompletion(
        service_id=service_id,
        deployment_name="gpt-4o-mini",
        endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    )
)

In [ ]:
# Define swarm agents as plugins
class TechSupportAgent:
    @kernel_function(name="handle_tech_support", description="Handle technical support questions")
    def handle(self, question: Annotated[str, "Technical question"]) -> str:
        return f"Tech Support: For '{question}', please restart your device and check for updates."

class BillingAgent:
    @kernel_function(name="handle_billing", description="Handle billing and payment questions")
    def handle(self, question: Annotated[str, "Billing question"]) -> str:
        return f"Billing: Regarding '{question}', your next billing date is next month. No charges pending."

class SalesAgent:
    @kernel_function(name="handle_sales", description="Handle sales and product inquiries")
    def handle(self, question: Annotated[str, "Sales question"]) -> str:
        return f"Sales: For '{question}', we have premium and basic plans. Premium offers advanced features."

# Add all swarm agents
kernel.add_plugin(TechSupportAgent(), plugin_name="tech_support")
kernel.add_plugin(BillingAgent(), plugin_name="billing")
kernel.add_plugin(SalesAgent(), plugin_name="sales")

In [ ]:
async def swarm_process(query: str) -> str:
    """Process query through swarm - agents collaborate and hand off as needed."""
    
    chat_history = ChatHistory()
    chat_history.add_system_message(
        "You coordinate a swarm of specialized agents (tech support, billing, sales). "
        "Route customer queries to the most appropriate agent based on the question type."
    )
    chat_history.add_user_message(query)
    
    execution_settings = AzureChatPromptExecutionSettings(
        service_id=service_id,
        function_choice_behavior=FunctionChoiceBehavior.Auto(),
    )
    
    chat_service = kernel.get_service(service_id)
    
    response = await chat_service.get_chat_message_content(
        chat_history=chat_history,
        settings=execution_settings,
        kernel=kernel,
    )
    
    return str(response)

In [ ]:
# Test swarm with different query types
queries = [
    "My app keeps crashing",
    "When is my next payment due?",
    "What features are in the premium plan?"
]

for query in queries:
    result = await swarm_process(query)
    print(f"Query: {query}")
    print(f"Response: {result}")
    print()

**Note:** In the swarm pattern with Semantic Kernel, the LLM intelligently selects the most appropriate specialized agent for each query. Agents can collaborate by the LLM calling multiple functions in sequence if needed.